# Updating data from GenBank

Author: Alexander Maksiaev

Purpose: Update labels from previously gotten data from GISAID + Andersen, using Genbank.

In [1]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

date_range = "01-01-2024--04-14-2025"
update_date = "05-21-2025"

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# originals = downloads + "GISAID_Andersen_Combined_Files/"
# temp_files = downloads + "Andersen_Temp_Files/"
# github_files = downloads + "Andersen_Downloads/avian-influenza/metadata/"
# complete = originals + date_range + "_B3_13_D1_1/D1_1/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Combinations/GISAID_Andersen/B3_13_D1_1/" 
temp_files = downloads + "Andersen/temp/"
github_files = downloads + "Andersen/avian-influenza/metadata/"
complete = originals + "01-01-2024--04-14-2025_B3_13_D1_1/D1_1/"



os.chdir(complete)

## Collection Dates

In [2]:
# Upload saved data 
# os.chdir(temp_files + "saved/")
os.chdir(downloads)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

os.chdir(github_files)
metadata = pd.read_csv("SraRunTable_automated.csv") 
metadata = metadata.merge(metadata_normalized, how="outer")

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
metadata_genbank = metadata.merge(genbank_mapping, on="Run")
# metadata_genbank = pd.read_csv("metadata_genbank_4-18-2025.csv") # Since 1/1/2024
os.chdir(originals)

display(metadata_genbank)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,BioSample Accession,is_retracted,retraction_detection_date_utc,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name
0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,SRS21079812,False,NaN,SRR28752446_HA_cns.fa,Consensus_SRR28752446_HA_cns_threshold_0.5_qua...,SRR28752446,HA,PP740722.1,4,A/blackbird/Texas/24-008354-001/2024
1,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,SRS21079812,False,NaN,SRR28752446_MP_cns.fa,Consensus_SRR28752446_MP_cns_threshold_0.5_qua...,SRR28752446,MP,PP740723.1,7,A/blackbird/Texas/24-008354-001/2024
2,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,SRS21079812,False,NaN,SRR28752446_NA_cns.fa,Consensus_SRR28752446_NA_cns_threshold_0.5_qua...,SRR28752446,NaN,PP740724.1,6,A/blackbird/Texas/24-008354-001/2024
3,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,SRS21079812,False,NaN,SRR28752446_NP_cns.fa,Consensus_SRR28752446_NP_cns_threshold_0.5_qua...,SRR28752446,NP,PP740725.1,5,A/blackbird/Texas/24-008354-001/2024
4,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,SRS21079812,False,NaN,SRR28752446_NS_cns.fa,Consensus_SRR28752446_NS_cns_threshold_0.5_qua...,SRR28752446,NS,PP740726.1,8,A/blackbird/Texas/24-008354-001/2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81551,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,2025-02-17,...,SRS24712065,False,NaN,SRR33124777_NP_cns.fa,Consensus_SRR33124777_NP_cns_threshold_0.5_qua...,SRR33124777,NP,PV572754.1,5,A/cattle/NV/25-006535-001-original/2025
81552,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,2025-02-17,...,SRS24712065,False,NaN,SRR33124777_NS_cns.fa,Consensus_SRR33124777_NS_cns_threshold_0.5_qua...,SRR33124777,NS,PV572757.1,8,A/cattle/NV/25-006535-001-original/2025
81553,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,2025-02-17,...,SRS24712065,False,NaN,SRR33124777_PA_cns.fa,Consensus_SRR33124777_PA_cns_threshold_0.5_qua...,SRR33124777,PA,PV572752.1,3,A/cattle/NV/25-006535-001-original/2025
81554,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,2025-02-17,...,SRS24712065,False,NaN,SRR33124777_PB1_cns.fa,Consensus_SRR33124777_PB1_cns_threshold_0.5_qu...,SRR33124777,PB1,PV572751.1,2,A/cattle/NV/25-006535-001-original/2025


In [3]:
# Get geolocation for second state attribute

os.chdir(home + "/references/")
state_ref = pd.read_csv("states_ref.csv")
metadata_genbank["name_state"] = metadata_genbank["genbank_name"].apply(lambda x: x.split("/")[2].replace("_", " ")) # Get the name of the state
metadata_genbank["Geo_Location"] = metadata_genbank["name_state"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x, 'Country'].iloc[0] + "-" + x if x in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x, 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x, 'Abbreviation'].iloc[0] if x in state_ref["State"].values else x)

print(metadata_genbank["Geo_Location"])

0        USA-TX
1        USA-TX
2        USA-TX
3        USA-TX
4        USA-TX
          ...  
81551    USA-NV
81552    USA-NV
81553    USA-NV
81554    USA-NV
81555    USA-NV
Name: Geo_Location, Length: 81556, dtype: object


In [4]:
# no_updates = pd.DataFrame()
# no_updates_isolate = []

# Get only labels that have no states or collection dates, and update them

def update(file_name, update_date):
    updates = {}
    with open(file_name) as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if line[0] == ">": # It's a header
                header = line 
                collection_date = header.split("|")[-3]
                state = header.split("/")[2].replace("_", " ")
                header = header.replace(header.split("|")[-4] + "|", "") # remove previous state
                # state = state.replace(": ", "-")
                # geo_location_collection_date = state + "|" + collection_date
                # geo_location_collection_date = collection_date
                isolate = header.split("/")[3]
                sequence = lines[i + 1] # Sequence always comes in one line after header
                row = metadata_genbank[metadata_genbank["isolate"] == isolate]

                if "-" not in collection_date: # If there are no dashes, i.e. if it's just the year
                    # Find the correct collection date, if it exists
                
                    try: 
                        collection_date = row["Collection_Date"].values[0]
                    #     print(id)
                    #     geo_location_collection_date = search_collection_date(id, row) # Update unknown dates, if possible
                    except:
                        print("No date found for isolate", isolate)
                    #         # no_updates_isolate.append(isolate)

                # if state == "USA": # If we don't have a state

                # try: 
                # state_new = row["Geo_Location"]
                try:
                    state = row["Geo_Location"].values[0]
                    print(state)
                except:
                    state = str([state_ref.loc[state_ref["Abbreviation"] == state, 'Country'].iloc[0] + "-" + state if state in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == state, 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == state, 'Abbreviation'].iloc[0] if state in state_ref["State"].values else state][0])
                    
                    # no_updates_isolate.append(isolate)

                updates[header] = [state, collection_date, sequence]

        f.close()

    updates_df = pd.DataFrame.from_dict(updates, orient="index", columns=["state", "collection_date", "sequence"])
    updates_df["geo_location_collection_date"] = updates_df["state"] + "|" + updates_df["collection_date"]
    updates_df["header"] = updates_df.index
    updates_df = updates_df.reset_index()

    updated_file_name = ".".join(file_name.split(".")[:-1]) + "_" + date_range + "_" + update_date + "_update." + file_name.split(".")[-1]

    with open(updated_file_name, "w") as g:

        for i, row in updates_df.iterrows():
            header = row["header"]
            # print(header)
            # print(header.split("|")[-3])
            
            # if header.split("/")[2] == "USA":
            #     header = header.replace(header.split("/")[2], row["state"])
            
            # header = header.replace(str(header.split("|")[-3]), str(row["geo_location_collection_date"])) # Only the first instance is replaced
            # header = str(row["collection_date"]).join(header.rsplit(str(header.split("|")[-3]), 1))
            header = str(row["geo_location_collection_date"]).join(header.rsplit(str(header.split("|")[-3]), 1))
            g.write(header)
            g.write(row["sequence"])

        g.close()

    # no_updates["isolate"] = no_updates_isolate
    # no_updates.to_csv("not_updated.csv")


In [5]:
# Create files with updates

# os.chdir(originals)

# for dirpath, dirs, files in os.walk(originals + date_range + "_B3_13_D1_1/"): # Find the fasta file
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         update(file_name, update_date)
#     break 

os.chdir(complete)
# file = "8-newid_B3.13_APR14_NS_trim_codon_aln_n3471_FINAL.fasta"
# update(file, update_date)

for dirpath, dirs, files in os.walk(complete): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file)
        update(file_name, update_date)
    break 

No date found for isolate 25-002460-001
No date found for isolate 25-002634-002
No date found for isolate 002453-003
No date found for isolate 005883-002
No date found for isolate 003547-001
No date found for isolate 004498-005
No date found for isolate 007118-006
No date found for isolate 007118-086
No date found for isolate 009029-001
No date found for isolate 003926-003
USA-AR
No date found for isolate 25-006702-001
No date found for isolate 25-006936-002
No date found for isolate 25-006941-001
No date found for isolate 25-007667-001
No date found for isolate 25-007837-001
No date found for isolate 25-009180-001
No date found for isolate 25-009390-003
No date found for isolate 25-009430-001
No date found for isolate 25-009514-002
No date found for isolate 001479-001
No date found for isolate 003299-006
No date found for isolate 003582-001
No date found for isolate 003584-001
No date found for isolate 003899-001
No date found for isolate 003900-001
No date found for isolate 004131-00

In [6]:
# search_collection_date_term("PP752829.1", metadata_genbank)